# Extraction EDA

Run `uv run -m uni_rag_agent extract run` after inventory has populated `data/uni_rag.sqlite`, then use this notebook to inspect extraction yield, failures, chunk counts, source types, source-location coverage, and quick diagnostic plots.

Safety boundary: this notebook is read-only. It reads generated app data only, opens SQLite in read-only mode, enables `PRAGMA query_only=ON`, and must not mutate SQLite, `Courses`, or any course file. Plots use pandas and matplotlib over already-generated app tables.

In [ ]:
from pathlib import Path
import sqlite3

import matplotlib.pyplot as plt
import pandas as pd

try:
    from IPython.display import Markdown, display
except ImportError:  # Allows helper cells to run outside Jupyter smoke checks.
    Markdown = None
    display = print


def display_md(text: str) -> None:
    if Markdown is not None:
        display(Markdown(text))
    else:
        print(text)


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "context"
        ).is_dir():
            return candidate
    raise RuntimeError(
        "Could not find repo root. Start Jupyter from this project or notebooks/."
    )


def show_barh(
    series: pd.Series,
    title: str,
    xlabel: str,
    color: str = "#4C78A8",
    max_rows: int = 15,
) -> None:
    data = series.dropna()
    data = data[data > 0].sort_values(ascending=True).tail(max_rows)
    if data.empty:
        display_md("_No rows to plot._")
        return
    height = max(3.0, 0.35 * len(data) + 1.2)
    ax = data.plot(kind="barh", figsize=(9, height), color=color)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("")
    ax.bar_label(ax.containers[0], padding=3)
    ax.figure.subplots_adjust(left=0.35, right=0.92, top=0.88, bottom=0.15)
    plt.show()


pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 160)
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

repo_root = find_repo_root(Path.cwd().resolve())
sqlite_path = repo_root / "data" / "uni_rag.sqlite"
if not sqlite_path.is_file():
    raise FileNotFoundError(
        f"Extraction database not found at {sqlite_path}. Run "
        "`uv run -m uni_rag_agent storage init`, then "
        "`uv run -m uni_rag_agent inventory run`, then "
        "`uv run -m uni_rag_agent extract run`."
    )
sqlite_uri = sqlite_path.resolve().as_uri() + "?mode=ro"

connection = sqlite3.connect(sqlite_uri, uri=True, timeout=5)
_ = connection.execute("PRAGMA query_only=ON")
display_md(f"**SQLite database:** `{sqlite_path}`")

## Plotting Helpers

The plots below are quick diagnostics over generated SQLite tables. They use pandas plus matplotlib and should remain lightweight enough to rerun after each extraction pass.

In [ ]:
latest_runs = pd.read_sql_query(
    """
    SELECT id, started_at, finished_at, status, config_json,
           files_seen, files_indexed, files_metadata_only, files_failed, error
    FROM extraction_runs
    ORDER BY id DESC
    LIMIT 20
    """,
    connection,
)
latest_runs

In [ ]:
run_metrics = latest_runs.sort_values("id").set_index("id")[
    ["files_indexed", "files_metadata_only", "files_failed"]
]
if run_metrics.empty:
    display_md("_No extraction run rows to plot._")
else:
    ax = run_metrics.plot(
        kind="bar",
        stacked=True,
        figsize=(9, 4),
        color=["#54A24B", "#B279A2", "#E45756"],
    )
    ax.set_title("Run outcomes by extraction_runs row")
    ax.set_xlabel("run id")
    ax.set_ylabel("files")
    ax.legend(title="metric")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

In [ ]:
extraction_docs = pd.read_sql_query(
    """
    SELECT extracted_documents.id AS extracted_document_id,
           files.id AS file_id,
           courses.name AS course_name,
           files.relative_path,
           files.extension,
           files.category,
           files.index_status,
           files.reason_not_indexed,
           extracted_documents.extractor_name,
           extracted_documents.status AS extraction_status,
           extracted_documents.text_length,
           extracted_documents.chunk_count,
           extracted_documents.error,
           COALESCE(extracted_documents.error, files.reason_not_indexed) AS failure_reason,
           extracted_documents.extracted_at
    FROM extracted_documents
    JOIN files ON files.id = extracted_documents.file_id
    LEFT JOIN courses ON courses.id = files.course_id
    ORDER BY extracted_documents.extracted_at DESC, files.relative_path
    """,
    connection,
)
extraction_docs.head(25)

In [ ]:
coverage_by_category = pd.read_sql_query(
    """
    SELECT files.category,
           files.extension,
           files.index_status,
           COUNT(*) AS file_count
    FROM files
    WHERE files.category IN ('document', 'slides', 'notebook', 'code', 'transcript')
    GROUP BY files.category, files.extension, files.index_status
    ORDER BY files.category, files.extension, files.index_status
    """,
    connection,
)
coverage_by_category

In [ ]:
coverage_pivot = coverage_by_category.pivot_table(
    index="category",
    columns="index_status",
    values="file_count",
    aggfunc="sum",
    fill_value=0,
)
if coverage_pivot.empty:
    display_md("_No category coverage rows to plot._")
else:
    ax = coverage_pivot.plot(
        kind="bar",
        stacked=True,
        figsize=(9, 4),
        color=["#54A24B", "#E45756", "#F58518", "#72B7B2", "#B279A2"],
    )
    ax.set_title("Extractable file status by category")
    ax.set_xlabel("category")
    ax.set_ylabel("files")
    ax.legend(title="index_status")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

In [ ]:
chunks = pd.read_sql_query(
    """
    SELECT chunks.id AS chunk_id,
           files.id AS file_id,
           courses.name AS course_name,
           files.relative_path,
           files.extension,
           chunks.source_type,
           chunks.location_type,
           chunks.location_value,
           chunks.token_count,
           LENGTH(chunks.text) AS text_chars
    FROM chunks
    JOIN files ON files.id = chunks.file_id
    LEFT JOIN courses ON courses.id = files.course_id
    ORDER BY files.relative_path, chunks.chunk_index
    """,
    connection,
)
chunks.head(25)

In [ ]:
chunk_source_summary = (
    chunks.groupby(["source_type", "location_type"], dropna=False)
    .agg(
        chunk_count=("chunk_id", "count"),
        median_tokens=("token_count", "median"),
        max_tokens=("token_count", "max"),
    )
    .reset_index()
    .sort_values(["source_type", "location_type"])
)
chunk_source_summary

In [ ]:
show_barh(
    chunks["source_type"].fillna("(missing source_type)").value_counts(),
    "Chunk count by source type",
    "chunks",
    color="#4C78A8",
)
show_barh(
    chunks["location_type"].fillna("(missing location_type)").value_counts(),
    "Chunk count by location type",
    "chunks",
    color="#72B7B2",
)

In [ ]:
indexed_docs = extraction_docs.loc[
    extraction_docs["extraction_status"] == "indexed"
].copy()
text_lengths = indexed_docs.loc[indexed_docs["text_length"] > 0, "text_length"]
token_counts = chunks.loc[chunks["token_count"].fillna(0) > 0, "token_count"]

if text_lengths.empty and token_counts.empty:
    display_md("_No positive text-length or token-count values to plot._")
else:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    if text_lengths.empty:
        axes[0].text(0.5, 0.5, "No text_length rows", ha="center", va="center")
        axes[0].set_axis_off()
    else:
        text_lengths.plot(kind="hist", bins=40, ax=axes[0], color="#4C78A8")
        axes[0].set_xscale("log")
        axes[0].set_title("Indexed document text length")
        axes[0].set_xlabel("characters, log scale")
    if token_counts.empty:
        axes[1].text(0.5, 0.5, "No token_count rows", ha="center", va="center")
        axes[1].set_axis_off()
    else:
        token_counts.plot(kind="hist", bins=40, ax=axes[1], color="#F58518")
        axes[1].set_xscale("log")
        axes[1].set_title("Chunk token counts")
        axes[1].set_xlabel("tokens, log scale")
    plt.tight_layout()
    plt.show()

In [ ]:
failed_extractions = extraction_docs.loc[
    extraction_docs["extraction_status"] == "failed"
].copy()
failed_extractions[
    [
        "course_name",
        "relative_path",
        "extension",
        "extractor_name",
        "failure_reason",
        "extracted_at",
    ]
].head(50)

## Failure Plots

These views make extractor failures easier to triage before rerunning with OCR, adding a converter, or fixing extractor-specific bugs.

In [ ]:
failure_reason_counts = (
    failed_extractions["failure_reason"]
    .fillna("(missing failure reason)")
    .value_counts()
)
show_barh(
    failure_reason_counts,
    "Failed extractions by reason",
    "files",
    color="#E45756",
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)
plot_specs = [
    (
        failed_extractions["extension"]
        .fillna("(missing extension)")
        .value_counts()
        .sort_values(ascending=True)
        .tail(12),
        "Failed files by extension",
        "#F58518",
    ),
    (
        failed_extractions["extractor_name"]
        .fillna("(missing extractor)")
        .value_counts()
        .sort_values(ascending=True)
        .tail(12),
        "Failed files by extractor",
        "#E45756",
    ),
    (
        failed_extractions["course_name"]
        .fillna("(missing course)")
        .value_counts()
        .sort_values(ascending=True)
        .tail(12),
        "Failed files by course",
        "#B279A2",
    ),
]
for ax, (series, title, color) in zip(axes, plot_specs):
    if series.empty:
        ax.text(0.5, 0.5, "No rows", ha="center", va="center")
        ax.set_axis_off()
        continue
    series.plot(kind="barh", ax=ax, color=color)
    ax.set_title(title)
    ax.set_xlabel("files")
    ax.set_ylabel("")
plt.show()